# 03 Long recordings and custom features

Two things that come up once the quickstart works: a recording too long to summarise with one vector, and a feature AFX does not ship.

In [ ]:
import os
import pickle
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

import afx_audio as afx
import demo_data

SR = 16000
tmp = Path(tempfile.mkdtemp(prefix='afx_long_'))

## A long recording

Eight 4 s events back to back, written to a WAV file. With a labelled dataset configured the events are dataset clips, otherwise synthetic ones; the event boundaries are kept as ground truth.

In [ ]:
SEG = 4.0
labelled = demo_data.labelled_files(n_classes=4, n_per_class=2)
events, labels = [], []
if labelled:
    for path, label in zip(*labelled):
        x, _ = afx.load_audio(path, sr=SR, duration=SEG)
        events.append(np.pad(x, (0, int(SEG * SR) - x.size)))
        labels.append(label)
else:
    for i in range(8):
        kind = demo_data.SYNTH_CLASSES[i % 4]
        events.append(demo_data.synth_clip(kind, SEG, SR, seed=i))
        labels.append(kind)

recording = np.concatenate(events)
long_path = tmp / 'recording.wav'
sf.write(long_path, recording, SR)
print(f'{long_path.name}: {sf.info(long_path).duration:.0f} s, events: {labels}')

`load_audio` with `offset` and `duration` reads one window without decoding the rest of the file, which matters for hour-long recordings.

In [ ]:
window, _ = afx.load_audio(long_path, sr=SR, offset=12.0, duration=4.0)
print(window.shape, 'event 4 of 8:', labels[3])
np.testing.assert_allclose(window, recording[12 * SR:16 * SR], atol=1e-4)

## One vector per segment

`extract_segments(signal, sr, config, segment_length, hop)` cuts the signal into fixed-length segments (seconds), runs `extract_all_features` on each and returns `(names, X, starts)` with one row per segment.

In [ ]:
config = {
    'features': {
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 512},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 512},
        'rms_energy': {'frame_size': 1024, 'hop_length': 512},
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
    },
    'aggregation': 'summary',
}
names, X, starts = afx.extract_segments(recording, SR, config, segment_length=2.0, hop=1.0)
print('X:', X.shape, ' segments start at', starts[:5], '...')

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
centres = starts + 1.0
for col, a in zip(['spectral_centroid_mean', 'spectral_flatness_mean'], ax):
    a.plot(centres, X[:, names.index(col)], 'o-')
    a.set_ylabel(col)
    for k, label in enumerate(labels):
        a.axvline(k * SEG, color='grey', lw=0.5)
        if a is ax[0]:
            a.text(k * SEG + 0.1, a.get_ylim()[1], label, va='top', fontsize=8)
ax[-1].set_xlabel('segment centre (s)')
fig.tight_layout()

Segments that straddle an event boundary mix both events. A shorter `segment_length` or `hop` trades resolution for noisier statistics.

## A custom feature

`register_feature(name, func)` makes a function available in configs, `FEATURE_MAP`, `feature_info` and every entry point. `func(signal, sr, **params)` returns `{name: per-frame array}`; its keyword parameters become the feature's config parameters and are validated like the built-in ones.

In [ ]:
def band_energy_ratio(signal, sr, *, split_hz=1000.0, frame_size=1024, hop_length=512):
    """Fraction of frame energy above split_hz."""
    frames = np.lib.stride_tricks.sliding_window_view(signal, frame_size)[::hop_length]
    power = np.abs(np.fft.rfft(frames * np.hanning(frame_size), axis=-1)) ** 2
    freqs = np.fft.rfftfreq(frame_size, 1 / sr)
    total = power.sum(axis=-1)
    with np.errstate(invalid='ignore', divide='ignore'):
        ratio = power[:, freqs >= split_hz].sum(axis=-1) / total
    return {'band_energy_ratio': np.where(total > 0, ratio, np.nan)}


afx.register_feature('band_energy_ratio', band_energy_ratio, units='ratio')
print(afx.feature_info('band_energy_ratio'))

cfg = {'features': {'band_energy_ratio': {'split_hz': 2000.0}, 'zcr': {}}, 'aggregation': 'mean'}
print(afx.extract_all_features(recording[:SR], SR, cfg))
try:
    afx.validate_config({'features': {'band_energy_ratio': {'split': 2000}}}, require_sample_rate=False)
except ValueError as e:
    print('ValueError:', e)

## Worker processes

`extract_features_from_folder(..., n_jobs=N)` and `AFXFeatureExtractor(n_jobs=N)` hand registered features to their worker processes, which pickles the function by reference: module plus name. A function defined in a notebook lives in `__main__`, which the workers cannot import: they start with forkserver (Linux) or spawn (Windows, macOS), never fork, so `extract_features_from_folder` rejects such a feature with a `ValueError`. Lambdas and nested functions never pickle.

In [ ]:
try:
    pickle.dumps(lambda signal, sr: {'x': signal[:1]})
except (pickle.PicklingError, AttributeError) as e:
    print(type(e).__name__, e)

The portable form keeps the function in an importable module. Here the module is written to the temp folder and put on `sys.path`; in a project it is just a file in the package.

In [ ]:
module_dir = tmp / 'mypkg'
module_dir.mkdir()
(module_dir / 'my_features.py').write_text(textwrap.dedent('''
    import numpy as np


    def band_energy_ratio(signal, sr, *, split_hz=1000.0, frame_size=1024, hop_length=512):
        """Fraction of frame energy above split_hz."""
        frames = np.lib.stride_tricks.sliding_window_view(signal, frame_size)[::hop_length]
        power = np.abs(np.fft.rfft(frames * np.hanning(frame_size), axis=-1)) ** 2
        freqs = np.fft.rfftfreq(frame_size, 1 / sr)
        total = power.sum(axis=-1)
        with np.errstate(invalid='ignore', divide='ignore'):
            ratio = power[:, freqs >= split_hz].sum(axis=-1) / total
        return {'band_energy_ratio': np.where(total > 0, ratio, np.nan)}
'''))
sys.path.insert(0, str(module_dir))
from my_features import band_energy_ratio as importable_ber

afx.register_feature('band_energy_ratio', importable_ber, units='ratio', overwrite=True)
pickle.dumps(importable_ber)
print('picklable:', importable_ber.__module__ + '.' + importable_ber.__qualname__)

A script that starts workers must create them under `if __name__ == '__main__':`. Each worker imports the script, and unguarded top-level code would run again in every worker. The script below sets spawn, the strictest start method, so it behaves the same on every OS.

In [ ]:
audio_dir = tmp / 'clips'
demo_data.write_synthetic_folder(str(audio_dir), n_per_class=2, seconds=1.0, sr=SR)

script = tmp / 'extract.py'
script.write_text(textwrap.dedent(f'''
    import multiprocessing

    import afx_audio as afx
    from my_features import band_energy_ratio

    afx.register_feature('band_energy_ratio', band_energy_ratio, units='ratio')
    CONFIG = {{'sample_rate': {SR}, 'features': {{'band_energy_ratio': {{}}, 'zcr': {{}}}}}}

    if __name__ == '__main__':
        multiprocessing.set_start_method('spawn')
        results = afx.extract_features_from_folder(
            {str(audio_dir)!r}, CONFIG, {str(tmp / 'features.csv')!r}, n_jobs=2, verbose=False)
        print(len(results), 'files, n_jobs=2 under spawn')
'''))
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(module_dir), os.environ.get('PYTHONPATH', '')])}
run = subprocess.run([sys.executable, str(script)], capture_output=True, text=True, env=env, cwd=tmp)
print(run.stdout or run.stderr)
print((tmp / 'features.csv').read_text().splitlines()[:3])

`unregister_feature` removes a registration (and restores a built-in feature it replaced).

In [ ]:
afx.unregister_feature('band_energy_ratio')
print('band_energy_ratio' in afx.FEATURE_MAP)

In [ ]:
import shutil

sys.path.remove(str(module_dir))
shutil.rmtree(tmp)